# Estimation of regression coefficients via GWR

In [ ]:
# Adding necessary libraries to Google Colab environment

# Installing the 'contextily' library for adding basemaps to plots.
# This library is useful for adding contextual geographical layers,
# such as maps, to visualizations. The '-q' flag is used to suppress
# unnecessary output during installation.
!pip install contextily -q

# Installing the 'splot' library for spatial data visualization.
# 'splot' works with PySAL to visualize spatial data, including spatial
# autocorrelation and cluster analysis. The '-q' flag is used to keep
# the output concise.
!pip install splot -q

# Installing additional libraries for spatial analysis and visualization.
# 'mapclassify' provides classification schemes for choropleth mapping,
# which can be useful for grouping spatial data into classes. This helps
# to visualize variations across different spatial regions more effectively.
!pip install mapclassify -q  # Library for choropleth mapping

# Installing the 'mgwr' library for performing geographically weighted
# regression (GWR). GWR is useful for understanding spatially varying
# relationships between variables by estimating local regression models.
# The '-q' flag is used to reduce verbosity during the installation process.
!pip install mgwr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.9/379.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.4/215.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.5 MB/s eta 0:00:00


In [ ]:
# Load libraries

# Library for plotting graphs and visualizations
import matplotlib.pyplot as plt

# Library for handling geospatial data
import geopandas as gpd

# Import GWR and MGWR for Geographically Weighted Regression
from mgwr.gwr import GWR, MGWR

# Import Sel_BW for selecting optimal bandwidth for GWR/MGWR
from mgwr.sel_bw import Sel_BW

# Suppress warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

In [ ]:
##################################################################
# Application: Geographically Weighted Regression (GWR) Analysis
# Description: Load Indonesia GeoJSON data, prepare variables, select
#              the optimal bandwidth, and fit the GWR model.
# URL:         https://bit.ly/MGWRapp3
##################################################################

# 1. Load Data
# -------------------------------------
gdf = gpd.read_file(
    "https://github.com/quarcs-lab/data-quarcs/raw/refs/heads/master/"
    "indonesia514/gdfBeta.geojson"
)

# 2. Prepare Variables
# -------------------------------------
# - Dependent Variable: GDP growth ('g')
y = gdf["g"].values.reshape((-1, 1))  # Reshape to a column vector

# - Independent Variable: Log GDP per capita (2010)
X = gdf[["ln_gdppc2010"]].values  # Ensure X is a 2D array

# - Spatial Coordinates: X and Y positions
u = gdf["COORD_X"]  # X-coordinates
v = gdf["COORD_Y"]  # Y-coordinates
coords = list(zip(u, v))  # List of tuples representing each location

# 3. Bandwidth Selection for GWR
# -------------------------------------
# - Select the optimal bandwidth controlling the spatial influence
gwr_selector = Sel_BW(coords, y, X, spherical=True)
gwr_bw = gwr_selector.search()

# 4. Fit the GWR Model
# -------------------------------------
# - Fit the model using the optimal bandwidth and spatial data
gwr_results = GWR(coords, y, X, gwr_bw).fit()

# 5. Display Results
# -------------------------------------
gwr_results.summary()


Model type                                                         Gaussian
Number of observations:                                                 514
Number of covariates:                                                     2

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                             41.453
Log-likelihood:                                                     -82.296
AIC:                                                                168.592
AICc:                                                               170.639
BIC:                                                              -3154.565
R2:                                                                   0.214
Adj. R2:                                                              0.212

Variable                              Est.         SE  t(Est/SE)    p-value
------------------------------- ---------- ---------- ------